# 05 · Query-based detector: RT-DETR

RT-DETR-L (Ultralytics, COCO-pretrained): a CNN backbone and hybrid encoder, then a transformer decoder whose
object queries are initialised from the top-scoring encoder features. Each query predicts at most one object;
training matches queries to ground truth one-to-one (Hungarian matching), so there are no anchors and no NMS.
To make the budget comparable with the CNN detectors, training is limited by GPU time (`time=` in Ultralytics,
which also fits the learning-rate schedule to that time).

In [ ]:
!pip install -q ultralytics
import os, urllib.request
# helper modules shared by all notebooks (data split, metric, submission format, torchvision training)
REPO = "https://raw.githubusercontent.com/MakcDutch/AI-homework2/main/"
for f in ["common.py", "tv_train.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from ultralytics import RTDETR
from ultralytics.utils import ROOT as ULTRA_ROOT
import common as C

ROOT = C.find_data_root()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Protocol
Same frames, masking, input size and metric as the other notebooks. Budget: `TIME_HOURS` of training on one T4,
set to the training time of the YOLOv8s baseline.

In [ ]:
TRAIN_STEP = 5
IMGSZ = 1024
TIME_HOURS = 1.5
BATCH = 4
DEVICE = 0
DATA_DIR = "/tmp/yolo_data"

# decoder with more queries: dense frames have ~46 objects on average and some have several hundred (01_eda §4)
base_cfg = (ULTRA_ROOT / "cfg/models/rt-detr/rtdetr-l.yaml").read_text()
open("rtdetr-l-q500.yaml", "w").write(base_cfg.replace("RTDETRDecoder, [nc]]", "RTDETRDecoder, [nc, 256, 500]]"))

EXPERIMENTS = {
    "rtdetr_base": dict(model="rtdetr-l.pt"),                                       # 300 queries
    "rtdetr_q500": dict(model="rtdetr-l-q500.yaml", weights="rtdetr-l.pt"),         # 500 queries
    "rtdetr_frozen_backbone": dict(model="rtdetr-l.pt", train_args=dict(freeze=10)),  # backbone lr = 0, only encoder/decoder learn
}
RUN = ["rtdetr_base"]

In [ ]:
data_yaml = C.build_yolo_dataset(ROOT, DATA_DIR, train_step=TRAIN_STEP, val_step=5)
val_images = C.list_frames(ROOT, "val")["image"].tolist()
test_images = C.list_frames(ROOT, "test")["image"].tolist()

## Training, evaluation and submission

In [ ]:
def run_experiment(name, cfg):
    model = RTDETR(cfg["model"])
    if "weights" in cfg:
        model = model.load(cfg["weights"])
    t0 = time.time()
    model.train(data=data_yaml, imgsz=IMGSZ, time=TIME_HOURS, batch=BATCH, device=DEVICE, seed=C.SEED,
                workers=4, project=f"{WORK}/runs", name=name, exist_ok=True, verbose=False,
                **cfg.get("train_args", {}))
    train_hours = (time.time() - t0) / 3600
    run_dir = model.trainer.save_dir

    best = RTDETR(f"{run_dir}/weights/best.pt")
    # no NMS in RT-DETR: max_det caps the number of queries returned per frame
    val_preds, ms = C.ultralytics_predict(best, val_images, imgsz=IMGSZ, device=DEVICE, batch=8)
    metrics, per_class = C.coco_evaluate(val_preds, ROOT, "val")
    test_preds, _ = C.ultralytics_predict(best, test_images, imgsz=IMGSZ, device=DEVICE, batch=8)

    C.save_preds(val_preds, f"{WORK}/val_preds_{name}.pkl")
    C.save_preds(test_preds, f"{WORK}/test_preds_{name}.pkl")
    C.make_submission(test_preds, ROOT, f"{WORK}/sub_{name}.csv")
    pd.Series(per_class, name=name).to_csv(f"{WORK}/per_class_{name}.csv")
    row = dict(exp=name, family="query-based", run_dir=str(run_dir), imgsz=IMGSZ, budget_hours=TIME_HOURS,
               train_hours=round(train_hours, 2), ms_per_img=round(ms, 1), **metrics)
    C.results_table(f"{WORK}/results.csv", row)
    return row


for name in RUN:
    print(run_experiment(name, EXPERIMENTS[name]))

## Results

In [ ]:
results = C.results_table(f"{WORK}/results.csv")
display(results.drop(columns="run_dir"))
display(pd.concat([pd.read_csv(f"{WORK}/per_class_{n}.csv", index_col=0) for n in RUN], axis=1).round(3))

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, run_dir in results.set_index("exp")["run_dir"].items():
    log = pd.read_csv(f"{run_dir}/results.csv"); log.columns = log.columns.str.strip()
    ax.plot(log["epoch"], log["metrics/mAP50-95(B)"], marker="o", label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("val mAP50-95 (Ultralytics, val subset)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/rtdetr_curves.png", dpi=120); plt.show()

## Failure cases

In [ ]:
preds = C.load_preds(f"{WORK}/val_preds_{RUN[-1]}.pkl")
C.show_failures(preds, ROOT, f"{WORK}/failures_{RUN[-1]}.png")